Approximation of the sum of two lognormal random variables with a single
lognormal random variable using moment matching.

Let $X = \lambda e^{\mu_1 + \sigma_1 Z_1} + (1 - \lambda) e^{\mu_2 + \sigma_2 Z}$
where $\lambda \in [0, 1]$, $\mu_1, \mu_2 \in \mathbb{R}$, and $\sigma_1, \sigma_2 > 0$.

We approximate $X$ with a lognormal random variable $Y = e^{\mu_Y + \sigma_Y Z_Y}$.

Let $m_1 = E[X]$ and $m_2 = E[X^2]$ be the first two moments of $X$.

We have 

$$
    m_1 = \lambda e^{\mu_1 + \frac{\sigma_1^2}{2}} + (1 - \lambda) e^{\mu_2 + \frac{\sigma_2^2}{2}}
$$

and 

$$
    m_2 = \lambda^2 e^{2\mu_1 + 2\sigma_1^2} + (1 - \lambda)^2 e^{2\mu_2 + 2\sigma_2^2} + 2\lambda(1 - \lambda) e^{\mu_1 + \mu_2 + \frac{\sigma_1^2 + \sigma_2^2}{2}}.
$$

Matching moments, we get

$$
    \sigma_Y = \sqrt{\ln\left(\frac{m_2}{m_1^2}\right)}
$$

$$
    \mu_Y = \ln(m_1) - \frac{1}{2} \sigma_Y^2
$$



In [3]:
import numpy as np

In [4]:
lbd = 0.3
mu_1 = 0.1
mu_2 = 0.1
sig_1 = 0.2
sig_2 = 0.2

## Lognormal Approximation

In [ ]:
def sum_lognorm_single_lognorm_approx(lbd, mu_1, mu_2, sig_1, sig_2):
    """
    X = lbd * exp(mu_1 + sig_1 * Z) + (1 - lbd) * exp(mu_2 + sig_2 * Z), Z = N(0, 1)
    Approximate X by a lognormal Y = exp(mu_y + sig_y * Z)
    """

    # First moment of X
    m1_x = lbd * np.exp(mu_1 + 0.5 * sig_1**2) + (1 - lbd) * np.exp(
        mu_2 + 0.5 * sig_2**2
    )
    # Second moment of X
    m2_x = (
        lbd**2 * np.exp(2 * mu_1 + 2 * sig_1**2)
        + (1 - lbd) ** 2 * np.exp(2 * mu_2 + 2 * sig_2**2)
        + 2 * lbd * (1 - lbd) * np.exp(mu_1 + mu_2 + 0.5 * (sig_1**2 + sig_2**2))
    )

    # Parameters of the approximating lognormal
    sig_y = np.log(m2_x / m1_x**2) ** 0.5
    mu_y = np.log(m1_x) - 0.5 * sig_y**2

    # Moments of Y for verification
    m1_y = np.exp(mu_y + 0.5 * sig_y**2)
    m2_y = np.exp(2 * mu_y + 2 * sig_y**2)

    assert np.isclose(m1_x, m1_y), "First moments do not match!"
    assert np.isclose(m2_x, m2_y), "Second moments do not match!"

    return {
        "sig_y": sig_y,
        "mu_y": mu_y,
    }

In [9]:
params = lognorm_approx(lbd, mu_1, mu_2, sig_1, sig_2)

In [11]:
params["mu_y"], params["sig_y"]

(np.float64(0.10830277383656667), np.float64(0.1529524511960065))

In [12]:
# Verification by computing moments of the approximating lognormal
assert np.isclose(params["m1_x"], params["m1_y"])
assert np.isclose(params["m2_x"], params["m2_y"])

## Shifted Lognormal Approximation

In [ ]:
def sum_lognorm_shifted_lognorm_approx(lbd, mu_1, mu_2, sig_1, sig_2):
    """
    X = lbd * exp(mu_1 + sig_1 * Z) + (1 - lbd) * exp(mu_2 + sig_2 * Z)
    Approximate X by Y = c + exp(mu_y + sig_y * Z)
    Returns m1_x, m2_x, m3_x, m1_y, m2_y, m3_y
    """

    # ---------- moments of X ----------
    E1 = np.exp(mu_1 + 0.5 * sig_1**2)
    E2 = np.exp(mu_2 + 0.5 * sig_2**2)

    E1_2 = np.exp(2 * mu_1 + 2 * sig_1**2)
    E2_2 = np.exp(2 * mu_2 + 2 * sig_2**2)
    E12 = np.exp(mu_1 + mu_2 + 0.5 * (sig_1 + sig_2) ** 2)

    E1_3 = np.exp(3 * mu_1 + 4.5 * sig_1**2)
    E2_3 = np.exp(3 * mu_2 + 4.5 * sig_2**2)
    E1_2E2 = np.exp(2 * mu_1 + mu_2 + 0.5 * (2 * sig_1 + sig_2) ** 2)
    E1E2_2 = np.exp(mu_1 + 2 * mu_2 + 0.5 * (sig_1 + 2 * sig_2) ** 2)

    m1_x = lbd * E1 + (1 - lbd) * E2

    m2_x = lbd**2 * E1_2 + (1 - lbd) ** 2 * E2_2 + 2 * lbd * (1 - lbd) * E12

    m3_x = (
        lbd**3 * E1_3
        + (1 - lbd) ** 3 * E2_3
        + 3 * lbd**2 * (1 - lbd) * E1_2E2
        + 3 * lbd * (1 - lbd) ** 2 * E1E2_2
    )

    # ---------- infer shifted lognormal parameters ----------
    var_x = m2_x - m1_x**2
    kappa3_x = m3_x - 3 * m1_x * m2_x + 2 * m1_x**3
    skew_x = kappa3_x / var_x**1.5

    # solve u^3 + 3u = skew_x
    disc = np.sqrt(skew_x**2 / 4 + 1)
    u = np.cbrt(skew_x / 2 + disc) + np.cbrt(skew_x / 2 - disc)

    t = u**2 + 1
    sig_y = np.sqrt(np.log(t))
    mu_y = 0.5 * np.log(var_x / (t * (t - 1)))

    c_y = m1_x - np.exp(mu_y) * np.sqrt(t)

    # ---------- moments of Y ----------
    EW = np.exp(mu_y) * np.sqrt(t)
    EW2 = np.exp(2 * mu_y) * t**2
    EW3 = np.exp(3 * mu_y) * t ** (9 / 2)

    m1_y = c_y + EW
    m2_y = c_y**2 + 2 * c_y * EW + EW2
    m3_y = c_y**3 + 3 * c_y**2 * EW + 3 * c_y * EW2 + EW3

    assert np.isclose(m1_x, m1_y), "First moments do not match!"
    assert np.isclose(m2_x, m2_y), "Second moments do not match!"
    assert np.isclose(m3_x, m3_y), "Third moments do not match!"

    return {
        "c_y": c_y,
        "mu_y": mu_y,
        "sig_y": sig_y,
    }

In [15]:
params = shifted_lognorm_approx(lbd, mu_1, mu_2, sig_1, sig_2)

In [16]:
assert np.isclose(params["m1_x"], params["m1_y"])
assert np.isclose(params["m2_x"], params["m2_y"])
assert np.isclose(params["m3_x"], params["m3_y"])

In [17]:
print(params["c"], params["mu_y"], params["sig_y"])

1.7208456881689926e-13 0.09999999999984158 0.2000000000000293
